# 32 — Prebuilt Agents

Load production-ready agent personas in one line. **40 agents** across **8 categories** — Architecture, Code Quality, Security, DevOps, Testing, Planning, Research, and Content.

**What you'll learn:**
1. Loading the built-in agent registry
2. Browsing agents by category
3. Searching for agents
4. Inspecting agent definitions
5. Building a live agent from a definition
6. Creating custom agent definitions
7. Merging registries and the `.shipit/agents/` override pattern

In [ ]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent
from shipit_agent.agents import AgentDefinition, AgentRegistry
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env("bedrock")

## 1. Loading the Built-in Registry

`AgentRegistry.default()` loads the 40 agents shipped with the framework. No configuration needed.

In [ ]:
# Load the built-in registry
registry = AgentRegistry.default()

print(f"Total agents: {len(registry)}")
print(f"Categories:   {registry.categories()}")
print(f"\nAll agents:")
for agent_def in registry.list_all():
    print(f"  [{agent_def.category:15s}] {agent_def.id:25s} — {agent_def.role[:60]}")

## 2. Browse Agents by Category

Filter agents by category to find what you need.

In [ ]:
# List all Security agents
print("=== Security Agents ===")
for a in registry.list_by_category("Security"):
    print(f"\n  {a.name}")
    print(f"  Role:  {a.role}")
    print(f"  Goal:  {a.goal}")
    print(f"  Tools: {a.tools}")

print("\n\n=== DevOps Agents ===")
for a in registry.list_by_category("DevOps"):
    print(f"\n  {a.name}")
    print(f"  Role: {a.role}")

## 3. Searching for Agents

Fuzzy search across name, role, goal, tags, and category.

In [ ]:
# Search for security-related agents
results = registry.search("security vulnerabilities audit")
print(f"Search 'security vulnerabilities audit' → {len(results)} results:")
for r in results[:5]:
    print(f"  {r.id:25s} — {r.role[:60]}")

print()
results = registry.search("code review python")
print(f"Search 'code review python' → {len(results)} results:")
for r in results[:5]:
    print(f"  {r.id:25s} — {r.role[:60]}")

### Category Statistics

Quick overview of how agents are distributed across categories.

In [ ]:
# Category distribution
print("Agent distribution by category:\n")
for cat in registry.categories():
    agents = registry.list_by_category(cat)
    tools_set = set()
    for a in agents:
        tools_set.update(a.tools)
    print(f"  {cat:20s} {len(agents):2d} agents | tools: {sorted(tools_set)}")

# Check a specific agent exists
print(f"\n'architect' in registry: {'architect' in registry}")
print(f"'nonexistent' in registry: {'nonexistent' in registry}")
print(f"repr: {repr(registry)}")

## 4. Inspecting an Agent Definition

Each agent has a rich definition with role, goal, backstory, tools, and a full system prompt.

In [ ]:
# Get a specific agent
agent_def = registry.get("security-auditor")

print(f"ID:          {agent_def.id}")
print(f"Name:        {agent_def.name}")
print(f"Role:        {agent_def.role}")
print(f"Goal:        {agent_def.goal}")
print(f"Backstory:   {agent_def.backstory[:200]}...")
print(f"Model:       {agent_def.model}")
print(f"Tools:       {agent_def.tools}")
print(f"Skills:      {agent_def.skills}")
print(f"Max Iter:    {agent_def.max_iterations}")
print(f"Category:    {agent_def.category}")
print(f"Tags:        {agent_def.tags}")

print("\n=== System Prompt (first 500 chars) ===")
print(agent_def.system_prompt()[:500])

## 5. Building a Live Agent from a Definition

Turn any `AgentDefinition` into a working `Agent` with built-in tools.

In [ ]:
# Build a live agent from the "researcher" definition
agent_def = registry.get("researcher")

agent = Agent.with_builtins(
    llm=llm,
    prompt=agent_def.system_prompt(),
    max_iterations=agent_def.max_iterations,
)

print(f"Agent built from '{agent_def.id}' with {len(agent.tools)} tools")

# Run the agent
result = agent.run("What are the top 3 trends in AI agents for 2026?")
display(Markdown(result.output[:800]))

### Run Multiple Prebuilt Agents — Category Showcase

Quick demo running agents from different categories on the same task.

In [ ]:
# Run agents from 3 different categories on the same question
question = "What are the security implications of using eval() in Python?"

for agent_id in ["security-auditor", "code-reviewer", "python-reviewer"]:
    defn = registry.get(agent_id)
    a = Agent(llm=llm, prompt=defn.system_prompt())
    result = a.run(question)
    print(f"\n{'='*60}")
    print(f"  [{defn.category}] {defn.name}")
    print(f"{'='*60}")
    print(result.output[:400])
    print("...")

### Serialization Round-Trip

Agent definitions can be serialized to JSON and back — useful for config files and APIs.

In [ ]:
import json

# Serialize to camelCase JSON
agent_def = registry.get("architect")
d = agent_def.to_dict()
print(f"Dict keys (camelCase): {list(d.keys())[:8]}...")

# Round-trip: to_dict → from_dict
restored = AgentDefinition.from_dict(d)
print(f"\nRound-trip check:")
print(f"  id match:       {agent_def.id == restored.id}")
print(f"  role match:     {agent_def.role == restored.role}")
print(f"  tools match:    {agent_def.tools == restored.tools}")
print(f"  prompt match:   {agent_def.prompt == restored.prompt}")

# Also works with snake_case keys
snake_dict = {"id": "test", "max_iterations": 12, "role": "tester"}
from_snake = AgentDefinition.from_dict(snake_dict)
print(f"\nFrom snake_case: id={from_snake.id}, max_iterations={from_snake.max_iterations}")

# Full JSON export
print(f"\nFull JSON (first 300 chars):")
print(json.dumps(d, indent=2)[:300])

## 6. Creating Custom Agent Definitions

Build your own agents with `AgentDefinition`.

In [ ]:
import json

custom_agent = AgentDefinition(
    id="my-data-pipeline-reviewer",
    name="Data Pipeline Reviewer",
    role="Senior Data Engineer specializing in ETL pipeline review",
    goal="Review data pipelines for correctness, performance, and reliability",
    backstory="15 years building data platforms at scale. Expert in Spark, Airflow, dbt.",
    model="sonnet",
    tools=["read_file", "grep_files", "glob_files", "bash"],
    max_iterations=10,
    prompt="""Review data pipelines with focus on:
1. Data quality checks — are there null checks, schema validation, dedup?
2. Idempotency — can the pipeline be safely re-run?
3. Error handling — what happens when upstream sources fail?
4. Performance — are there unnecessary full-table scans?
5. Monitoring — are there alerts for SLA breaches?

Output a structured review with findings and recommendations.""",
    category="Data Engineering",
    tags=["data", "pipeline", "etl", "review"],
)

print(f"Custom agent: {custom_agent.name}")
print(f"\nSystem prompt:\n{custom_agent.system_prompt()[:500]}")
print(f"\nJSON (first 400 chars):\n{json.dumps(custom_agent.to_dict(), indent=2)[:400]}...")

## 7. Merging Registries and `.shipit/agents/` Override

Combine built-in agents with project-specific ones. Project agents with the same ID override built-in agents.

**Directory pattern:**
```
.shipit/
  agents/
    my-custom-agent.json
    researcher.json          ← overrides built-in "researcher"
```

In [ ]:
import json, tempfile
from pathlib import Path

# Create a project-local registry
project_registry = AgentRegistry([custom_agent])

# Override the built-in "researcher" with a custom version
custom_researcher = AgentDefinition(
    id="researcher",  # same ID — will override built-in
    name="Research Specialist (Custom)",
    role="Domain-specific research agent for our fintech product",
    goal="Research financial regulations and compliance requirements",
    prompt="You are a fintech compliance researcher. Focus on SEC, FINRA, and EU regulations.",
    tools=["read_file", "web_search", "open_url"],
    category="Research",
)

merged = registry.merge(AgentRegistry([custom_researcher, custom_agent]))
print(f"Built-in: {len(registry)} agents → Merged: {len(merged)} agents")

r = merged.get("researcher")
print(f"\n'researcher' after merge: {r.name}")
print(f"  Role: {r.role}")

d = merged.get("my-data-pipeline-reviewer")
print(f"\nCustom agent found: {d.name}")

# Load from directory (simulates .shipit/agents/)
with tempfile.TemporaryDirectory() as tmpdir:
    agent_data = {"id": "compliance-checker", "name": "Compliance Checker",
                  "role": "Regulatory compliance specialist", "goal": "Check code for compliance",
                  "tools": ["read_file", "grep_files"], "prompt": "Review for GDPR and PCI-DSS.",
                  "category": "Compliance", "tags": ["compliance", "gdpr"]}
    with open(Path(tmpdir) / "compliance-checker.json", "w") as f:
        json.dump(agent_data, f)
    
    local = AgentRegistry.from_directory(tmpdir)
    full = registry.merge(local)
    print(f"\nFrom directory: {len(local)} agent → Total: {len(full)} agents")
    print(f"  {full.get('compliance-checker').name}: {full.get('compliance-checker').role}")

## 8. Using Prebuilt Agents with ShipCrew

Prebuilt agents integrate with `ShipCrew` for multi-agent orchestration. See notebook 33 for full examples.

In [ ]:
from shipit_agent.deep.ship_crew import ShipAgent, ShipTask, ShipCrew

researcher_def = registry.get("researcher")
writer_def = registry.get("blog-writer")

researcher_sa = ShipAgent(
    name="researcher",
    agent=Agent.with_builtins(llm=llm, prompt=researcher_def.system_prompt()),
    role=researcher_def.role,
    goal=researcher_def.goal,
    backstory=researcher_def.backstory,
)

writer_sa = ShipAgent(
    name="writer",
    agent=Agent.with_builtins(llm=llm, prompt=writer_def.system_prompt()),
    role=writer_def.role,
    goal=writer_def.goal,
    backstory=writer_def.backstory,
)

print(f"Researcher: {researcher_sa.role}")
print(f"Writer:     {writer_sa.role}")
print("\nReady for ShipCrew orchestration! See notebook 33.")

## Summary

| Feature | API |
|---------|-----|
| Load built-in agents | `AgentRegistry.default()` |
| Get agent by ID | `registry.get("security-auditor")` |
| Search agents | `registry.search("security audit")` |
| Browse by category | `registry.list_by_category("Security")` |
| All categories | `registry.categories()` |
| Custom definitions | `AgentDefinition(id=..., role=..., prompt=...)` |
| Merge registries | `registry.merge(other_registry)` |
| Load from directory | `AgentRegistry.from_directory(".shipit/agents/")` |
| Build live agent | `Agent.with_builtins(llm=llm, prompt=def.system_prompt())` |

**40 agents** across **8 categories** — ready to use out of the box.